# 01. 점포 매출 기술통계

- **분석 배경**
  - 점포 월 매출 데이터를 처음 받았을 때, 본격적인 분석에 앞서 데이터의 상태와 분포부터 파악하고자 합니다.
  - 월 매출에 입력 오류가 섞여 있을 수 있으므로 먼저 오류를 찾아 보정하고 이후 실습에서 쓸 수 있도록 저장합니다.
- **분석 대상**
  - 100개 점포의 2024년 1월 ~ 2025년 12월 월 매출입니다.
  - 점포 사이의 비교는 최근 1년인 2025년의 점포별 월평균 매출로 합니다.
- **분석 질문**
  - 대형마트 점포의 월 매출 수준은 어느 정도인가?
  - 점포 유형별로 얼마나 차이가 나는가?
- **사용 데이터**
  - `store_info.csv`(점포 마스터 100행)
  - `store_month.csv`(점포 × 월 2,400행)
- **분석 단위**: 점포 (점포별 2025년 월평균 매출, n = 100)

**학습 목표**

1. 기술통계로 입력 오류를 찾아내고, 분석 단위를 정할 수 있다.
2. 연속형 변수의 평균·중위수·표준편차·분위수·왜도로 분포의 모습을 설명할 수 있다.
3. 범주형 변수의 도수·상대도수·교차표를 만들 수 있다.
4. 표준화 점수로 단위나 규모가 다른 값을 비교할 수 있다.
5. 평균의 95% 신뢰구간을 전체와 점포 유형별로 계산하고 업무 언어로 해석할 수 있다.

## 0. 준비

### 라이브러리 불러오기

In [ ]:
# 필요한 모듈을 임포트합니다.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import hds

### 그래프 한글 폰트 설정

구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.<br>
음수 부호(−)가 네모로 깨지지 않도록 하는 설정도 함께 처리합니다.

In [ ]:
# 구글 폰트(고운바탕)를 내려받아 그래프 한글 폰트로 설정합니다.
# 처음 실행할 때만 인터넷 연결이 필요하고, 이후에는 내려받은 파일을 다시 사용합니다.
hds.plot.set_font(font_name='Gowun Batang')

# 그래프 제목을 굵게 표시합니다.
plt.rc(group='axes', titleweight='bold')

### 데이터 불러오기

In [ ]:
# 점포 마스터 csv 파일을 읽고 store_info에 할당합니다.
store_info = pd.read_csv(filepath_or_buffer='../data/store_info.csv', encoding='utf-8-sig')

# store_info의 처음 5행과 마지막 5행을 확인합니다.
# [참고] 데이터프레임 이름만 실행하면 처음 5행과 마지막 5행, 행 개수와 열 개수를 함께 보여줍니다.
store_info

In [ ]:
# store_info의 정보를 확인합니다.
store_info.info()

In [ ]:
# 점포 월 매출 csv 파일을 읽고 store_month에 할당합니다.
store_month = pd.read_csv(filepath_or_buffer='../data/store_month.csv', encoding='utf-8-sig')

# store_month의 처음 5행과 마지막 5행을 확인합니다.
store_month

In [ ]:
# store_month의 정보를 확인합니다.
store_month.info()

## 1. 두 테이블 결합과 기본 점검

**질문**: 점포 × 월 테이블에 점포 특성을 붙이면 행 수가 그대로 유지되는가? 결측값이나 중복 행은 없는가?

In [ ]:
# store_month에 store_id 기준으로 store_info를 결합하고 df에 할당합니다.
df = store_month.merge(right=store_info, how='left', on='store_id')

# 결합 전후 행 개수를 확인합니다.
print('결합 전 행 개수:', store_month.shape[0])
print('결합 후 행 개수:', df.shape[0])

In [ ]:
# 같은 점포·같은 월이 두 번 이상 나오는 중복 행 개수를 확인합니다.
print('점포 × 월 중복 행 개수:', df.duplicated(subset=['store_id', 'ym']).sum())

In [ ]:
# df의 열별 결측값 개수를 확인합니다.
df.isna().sum()

**해석**
- 결합 전후 행 수가 같으면 모든 월 매출 행이 점포 마스터와 정확히 하나씩 짝을 이룬 것입니다.
- 중복 행 수와 열별 결측값 수가 모두 0이면 형식상으로는 깨끗한 데이터입니다.
  - 형식이 깨끗하다고 해서 값까지 맞다는 것은 아닙니다.
- 다음 단계에서 값의 범위를 확인합니다.

## 2. 기술통계로 이상한 값 찾기

**질문**: 매출, 구매 건수, 객단가의 범위가 상식적인가?

In [ ]:
# 매출, 구매 건수, 객단가의 기술통계량을 확인합니다.
df.loc[:, ['sales', 'transactions', 'avg_ticket']].describe()

**해석**
- `avg_ticket`(객단가)의 최댓값을 75% 분위수(Q3)와 비교해 보세요.
- 대부분의 행은 몇만 원대인데, 최댓값만 자릿수가 다릅니다.
- 마트 객단가가 이 정도로 높을 수는 없으므로 이와 같은 경우에는 데이터의 입력 오류를 의심합니다.

**질문**: 객단가가 비정상적으로 큰 행은 어느 점포의 어느 달인가? 기준을 정해서 코드로 찾아봅니다.

In [ ]:
# 객단가의 1사분위수와 3사분위수를 q1, q3에 할당합니다.
q1, q3 = df['avg_ticket'].quantile(q=[0.25, 0.75])

# 1사분위수와 3사분위수를 확인합니다.
print(f'1사분위수: {q1:,.0f}  /  3사분위수: {q3:,.0f}')

In [ ]:
# 3사분위수에 사분범위의 1.5배를 더한 값을 극단값 기준으로 정하고 upper_limit에 할당합니다.
# [참고] 상자 그림에서 이 기준을 넘는 값을 극단값으로 표시하는 방식입니다.
upper_limit = q3 + 1.5 * (q3 - q1)

# 객단가의 극단값 상한 기준을 확인합니다.
print(f'객단가 극단값 상한 기준: {upper_limit:,.0f}원')

In [ ]:
# 객단가가 극단값 상한 기준을 초과하는지 여부를 is_error에 할당합니다.
# [참고] gt()는 greater than(>) 연산을 수행하고, 시리즈의 원소별로 True 또는 False를 반환합니다.
is_error = df['avg_ticket'].gt(other=upper_limit)

# 극단값 상한 기준을 초과하는 행을 확인합니다.
df.loc[is_error, ['store_id', 'ym', 'sales', 'transactions', 'avg_ticket']]

**해석**
- 기준을 넘는 행은 위 표에 나온 행뿐입니다.
- 행 번호를 눈으로 골라 지우지 않고 **기준을 코드로 정해 찾았기 때문에** 데이터가 바뀌어도 같은 방법으로 다시 찾을 수 있습니다.

**질문**: 이 값은 왜 생겼을까? 같은 점포의 앞뒤 달과 비교해 원인을 추정합니다.

In [ ]:
# 기준을 초과한 행의 점포 ID를 error_store에 할당합니다.
error_store = df.loc[is_error, 'store_id'].iloc[0]

# 입력 오류가 있는 점포의 행만 선택하고 same_store에 할당합니다.
same_store = df.loc[df['store_id'].eq(other=error_store), :]

# 같은 점포의 전체 기간 매출, 구매 건수, 객단가를 확인합니다.
same_store.loc[:, ['store_id', 'ym', 'sales', 'transactions', 'avg_ticket']]

**해석**
- S008 점포의 2024년 11월의 구매 건수는 다른 달과 비슷한데, 매출만 다른 달보다 몇 배나 큽니다.
- 손님 수는 그대로인데 매출만 몇 배로 뛸 수는 없으므로 **매출 단위를 잘못 입력한 것**으로 추정합니다.
- 실무에서는 원천 시스템 담당자에게 확인해 고치는 것이 원칙이고, 확인할 수 없으면 분석에서 제외합니다.
- 이번 실습에서는 **원천 자료를 확인한 결과 매출이 10배로 잘못 입력된 것으로 확인되었다고 가정**하고 값을 보정합니다.

**질문**: 매출을 보정하면 객단가는 어떻게 다시 계산해야 하는가?

In [ ]:
# df를 복사하고 df_clean에 할당합니다.
df_clean = df.copy()

# 입력 오류 행의 매출을 10으로 나누어 보정합니다.
df_clean.loc[is_error, 'sales'] = df_clean.loc[is_error, 'sales'] / 10

# 보정한 매출로 객단가를 다시 계산하고 new_ticket에 할당합니다.
# [참고] 객단가는 매출(백만 원)에 1,000,000을 곱해 원 단위로 바꾼 뒤 구매 건수로 나눈 값입니다.
new_ticket = df_clean.loc[is_error, 'sales'] * 1e6 / df_clean.loc[is_error, 'transactions']

# new_ticket을 출력합니다.
print(new_ticket)

In [ ]:
# 다시 계산한 객단가를 정수로 반올림해서 입력 오류 행의 avg_ticket에 할당합니다.
df_clean.loc[is_error, 'avg_ticket'] = new_ticket.round().astype(dtype=int)

# 보정 후 입력 오류가 있던 점포의 행만 선택하고 same_store에 할당합니다.
same_store = df_clean.loc[df_clean['store_id'].eq(other=error_store), :]

# 보정 후 같은 점포의 전체 기간 매출, 구매 건수, 객단가를 확인합니다.
same_store.loc[:, ['store_id', 'ym', 'sales', 'transactions', 'avg_ticket']]

**해석**
- 보정한 달의 매출과 객단가가 앞뒤 달과 비슷한 수준이 되었습니다.
- 객단가는 매출 ÷ 구매 건수로 만든 값이므로 틀린 매출을 그대로 둔 채 다시 나누면 같은 오류가 그대로 남습니다.
- **원인이 된 매출을 먼저 고치고, 그 값으로 객단가를 다시 계산**해야 합니다.

**질문**: 보정 후 행 수와 객단가 범위는 어떻게 바뀌는가?

In [ ]:
# 보정 전후 행 개수를 확인합니다.
print('보정 전:', df.shape[0], '행 → 보정 후:', df_clean.shape[0], '행')

In [ ]:
# 보정 후 매출, 구매 건수, 객단가의 기술통계량을 확인합니다.
df_clean.loc[:, ['sales', 'transactions', 'avg_ticket']].describe()

**해석**
- 행을 지우지 않고 값을 고쳤으므로 행 수는 그대로이고, 객단가 최댓값이 다른 행들과 비슷한 수준으로 내려왔습니다.
- 평균은 극단값 하나에도 끌려가므로 분석 전에 기술통계로 값의 범위를 확인하는 습관이 중요합니다.

**질문**: 보정한 데이터를 다른 실습에서 다시 쓸 수 있도록 저장합니다.

In [ ]:
# df_clean을 data 폴더에 pkl 파일로 저장합니다.
# [참고] pkl은 파이썬 객체를 그대로 저장하는 pickle 형식으로, 열의 자료형까지 그대로 유지됩니다.
df_clean.to_pickle(path='../data/store_month_clean.pkl')

# data 폴더에 있는 폴더명과 파일명 목록을 확인합니다.
sorted(os.listdir(path='../data'))

**해석**
- 원천 csv 파일은 그대로 두고, 결합과 보정을 마친 데이터를 별도의 pickle 파일로 저장했습니다.
- 다른 실습에서는 pickle 파일을 불러오므로 같은 전처리를 반복하지 않고, 모든 분석이 같은 데이터를 사용합니다.
  - pickle 파일은 읽을 때 그 안의 코드가 실행될 수 있으므로 직접 만들었거나 믿을 수 있는 파일만 읽습니다.
  - pandas 버전이 크게 다르면 읽지 못할 수도 있으므로 다른 부서와 데이터를 주고받을 때는 csv 파일을 사용합니다.

## 3. 분석 단위 정하기

**질문**: 점포 × 월 행을 그대로 써서 '점포 매출'의 평균과 신뢰구간을 계산해도 되는가?

In [ ]:
# 점포별 행 개수의 기술통계량을 확인합니다.
df_clean['store_id'].value_counts().describe()

**해석**
- 각 점포마다 24개월씩 반복해서 들어 있습니다.
  - 같은 점포의 매출은 매달 비슷한 수준이므로 이 행들은 **서로 독립인 관측값이 아닙니다.**
- 현재 데이터로 모평균의 95% 신뢰구간을 계산하면 표본이 실제보다 훨씬 많은 것처럼 계산되어 구간이 지나치게 좁아집니다.
  - 신뢰구간 계산에 사용되는 표준오차의 분모는 표본 크기의 제곱근이기 때문입니다.
- 점포별로 월평균 매출을 계산하여 24행을 1행으로 요약하는 테이블을 새로 만듭니다.

**질문**: 점포별 2025년 월평균 매출 테이블(점포 하나 = 한 행)을 만들면 몇 개 점포가 되는가?

In [ ]:
# df_clean에서 2025년 행만 선택하고 store_2025에 할당합니다.
store_2025 = df_clean.loc[df_clean['ym'].str.startswith('2025'), :]

# store_2025에서 점포별 월평균 매출과 월평균 객단가를 계산하고 store_2025에 재할당합니다.
store_2025 = store_2025.groupby(by='store_id') \
    .agg(sales=('sales', 'mean'), avg_ticket=('avg_ticket', 'mean'))

# store_2025에 점포 특성을 결합합니다.
store_2025 = store_2025.merge(right=store_info, on='store_id')

# store_2025의 행 개수와 열 개수를 확인합니다.
store_2025.shape

In [ ]:
# store_2025의 처음 5행과 마지막 5행을 확인합니다.
store_2025

**해석**
- 이제 각 행은 한 점포의 월평균 실적과 특성을 나타냅니다.
  - 각 행의 `sales`는 그 점포의 2025년 월평균 매출(백만 원), `avg_ticket`은 월평균 객단가(원)입니다.
- 이후 분석은 모두 이 100개 점포 테이블로 합니다.

## 4. 대푯값과 퍼짐 정도

**질문**: 점포 월평균 매출과 객단가의 평균, 중위수, 범위, 분산, 표준편차, 분위수는 얼마인가?

In [ ]:
# 매출과 객단가 열을 선택하고 sub에 할당합니다.
sub = store_2025.loc[:, ['sales', 'avg_ticket']]

# 열별 대푯값과 산포도를 한 표로 정리하고 summary에 할당합니다.
summary = pd.DataFrame(data={'평균': sub.mean(), '중위수': sub.median(),
                             '최솟값': sub.min(), '최댓값': sub.max(),
                             '범위': sub.max() - sub.min(),
                             '분산': sub.var(), '표준편차': sub.std(),
                             'Q1(25%)': sub.quantile(q=0.25),
                             'Q3(75%)': sub.quantile(q=0.75)})

# 행과 열을 바꾸고 소수점 첫째 자리까지 확인합니다.
summary.T.round(decimals=1)

**해석**
- 매출은 평균이 중위수보다 큽니다.
  - 이는 매출이 아주 큰 소수 점포가 전체 평균을 위로 끌어올렸다는 신호입니다.
- 분산은 단위가 '백만 원의 제곱'이라 해석하기 어렵습니다.
  - 분산 대신 제곱근을 취해 같은 단위로 되돌린 **표준편차**를 주로 씁니다.
- Q1 ~ Q3 사이에 가운데 50% 점포가 들어 있습니다.
  - 사분(위)범위는 극단값의 영향을 덜 받는 퍼짐 지표입니다.

**질문**: 점포 × 월 매출의 범위는 2024년과 2025년에 어떻게 다른가? 표준편차와 함께 비교합니다.

In [ ]:
# df_clean에 연도 열을 추가합니다.
df_clean['year'] = df_clean['ym'].str[:4]

# 연도별 매출의 최솟값, 최댓값, 표준편차를 계산하고 year_spread에 할당합니다.
year_spread = df_clean.groupby(by='year')['sales'].agg(func=['min', 'max', 'std'])

# 최댓값에서 최솟값을 뺀 범위를 계산합니다.
year_spread['range'] = year_spread['max'] - year_spread['min']

# year_spread를 소수점 첫째 자리까지 반올림해서 확인합니다.
year_spread.round(decimals=1)

**해석**
- 범위는 연도별로 차이가 크지만, 표준편차는 거의 비슷합니다.
  - 범위는 최댓값과 최솟값 **두 값만으로** 정해지므로 극단값 하나에 크게 흔들립니다.
  - 표준편차를 계산할 때 모든 값을 반영합니다.
- 앞에서 입력 오류를 보정하지 않았다면 2024년 범위는 훨씬 더 커졌을 것입니다.

**질문**: 점포 월평균 매출을 10% 단위로 나누면(10분위수), 상위 10% 점포의 기준선은 얼마인가?

In [ ]:
# 0.1부터 0.9까지 0.1 간격의 확률을 probs에 할당합니다.
# [참고] np.linspace()는 시작값과 끝값을 모두 포함하며, 두 값 사이를 균등하게 나눈 수열을 생성합니다.
probs = np.linspace(start=0.1, stop=0.9, num=9).round(decimals=1)

# probs를 출력합니다.
print(probs)

In [ ]:
# 점포 월평균 매출의 10분위수를 계산하고 deciles에 할당합니다.
deciles = store_2025['sales'].quantile(q=probs)

# deciles를 소수점 첫째 자리까지 반올림해서 확인합니다.
deciles.round(decimals=1)

In [ ]:
# 90% 지점을 10% 지점으로 나누어 상하위 격차 배율을 계산합니다.
print('P90 / P10:', round(number=deciles[0.9] / deciles[0.1], ndigits=2))

**해석**
- 10분위수의 0.5 지점은 중위수이고, 0.9 지점은 '이보다 매출이 큰 점포가 10%'라는 상위 10% 기준선입니다.
  - 근로자 연봉을 '중위 연봉', '상위 10% 연봉'으로 발표하는 것과 같은 방식입니다.
- P90 / P10은 상위 10% 경계가 하위 10% 경계의 몇 배인지를 나타내는 격차 지표입니다.
  - 평균과 달리 양 끝 극단값의 영향을 받지 않습니다.

**질문**: 범주형 변수인 점포 유형은 범주별로 점포가 몇 개(도수)이고, 전체에서 몇 %(상대도수)인가?

In [ ]:
# 점포 유형의 고유값을 확인합니다.
store_2025['store_type'].unique()

In [ ]:
# 점포 유형을 서열형으로 변환합니다.
# [참고] 서열형 변수는 고유값별로 순위를 유지하므로 오름차순 또는 내림차순 정렬한 결과를 통제하기 쉽습니다.
store_2025['store_type'] = pd.Categorical(values=store_2025['store_type'], categories=['소형', '중형', '대형'], ordered=True)

# store_2025의 열별 자료형을 확인합니다.
store_2025.dtypes

In [ ]:
# 점포 유형별 도수를 type_freq에 할당합니다.
# [참고] value_counts()는 시리즈의 고유값별 도수를 계산합니다.
type_freq = store_2025['store_type'].value_counts()

# 점포 유형별 상대도수를 type_prop에 할당합니다.
# [참고] normalize=True는 상대도수를 계산합니다.
type_prop = store_2025['store_type'].value_counts(normalize=True)

# 점포 유형별 도수와 상대도수를 한 표로 정리합니다.
pd.DataFrame(data={'도수': type_freq, '상대도수': type_prop}).sort_index().round(decimals=3)

**해석**
- 범주형 변수에서 범주(고유값)별로 개수를 센 것을 도수라 합니다.
  - 도수가 가장 큰 고유값이 범주형 변수의 **최빈값**입니다.
- 상대도수는 도수를 전체 개수로 나눈 값이므로 합계가 1이 됩니다.
  - 전체 개수가 다른 두 집단을 비교할 때는 도수보다 상대도수를 봅니다.

**질문**: 점포 유형과 지역을 함께 보면, 지역마다 유형 구성이 어떻게 다른가?

In [ ]:
# 점포 유형과 지역의 교차표를 생성합니다.
pd.crosstab(index=store_2025['store_type'], columns=store_2025['region'], margins=True, margins_name='합계')

In [ ]:
# 지역별로 점포 유형의 상대도수를 계산합니다.
# [참고] normalize='columns'는 열(지역)마다 합계가 1이 되도록 나눕니다.
pd.crosstab(index=store_2025['store_type'], columns=store_2025['region'], normalize='columns', margins=True, margins_name='합계') \
    .round(decimals=3)

**해석**
- 교차표는 두 범주형 변수의 조합별 도수를 한눈에 보여 줍니다.
  - `margins=True`를 추가하면 행과 열별 합계를 함께 출력합니다.
  - `margins_name`으로 합계 이름을 정할 수 있습니다.(기본값: 'All')
- 지역마다 점포 수가 다르므로 지역 사이의 유형 구성을 비교할 때는 열 기준 상대도수를 봅니다.
- 가설검정 실습에서는 카이제곱 검정을 실행하기 전에 교차표를 출력하고, 행 또는 열 기준 상대도수가 크게 다른지 확인합니다.

**질문**: 연속형 값은 같은 값이 거의 반복되지 않으므로 구간으로 묶어서 가장 많은 점포가 몰린 구간(최빈 구간)을 찾습니다.

In [ ]:
# 매출의 기술통계량을 확인합니다.
store_2025['sales'].describe().round(decimals=1)

In [ ]:
# 매출을 500(백만 원) 간격으로 구간화하고 sales_bins에 할당합니다.
# [참고] pd.cut()은 연속형 변수를 일정한 간격으로 구분한 범주형 변수로 변환합니다.
sales_bins = pd.cut(x=store_2025['sales'], bins=range(0, 10001, 500))

# sales_bins의 범주별 도수와 상대도수를 계산한 도수분포표를 확인합니다.
# [참고] 도수분포표를 시각화한 것을 히스토그램이라고 합니다.
sales_freq = sales_bins.value_counts()
sales_prop = sales_bins.value_counts(normalize=True)
pd.concat(objs=[sales_freq, sales_prop], axis=1).sort_index()

In [ ]:
# 도수가 가장 큰 매출 구간을 확인합니다.
print('매출 최빈 구간(백만 원):', sales_bins.value_counts().idxmax())

In [ ]:
# 객단가의 기술통계량을 확인합니다.
store_2025['avg_ticket'].describe().round(decimals=1)

In [ ]:
# 객단가를 2,000원 간격으로 구간화하고 ticket_bins에 할당합니다.
ticket_bins = pd.cut(x=store_2025['avg_ticket'], bins=range(40000, 64001, 2000))

# ticket_bins의 범주별 도수와 상대도수를 계산한 도수분포표를 확인합니다.
ticket_freq = ticket_bins.value_counts()
ticket_prop = ticket_bins.value_counts(normalize=True)
pd.concat(objs=[ticket_freq, ticket_prop], axis=1).sort_index()

In [ ]:
# 도수가 가장 큰 객단가 구간을 확인합니다.
print('객단가 최빈 구간(원):', ticket_bins.value_counts().idxmax())

**해석**
- 연속형 변수는 셀 수 없거나, 고유값이 매우 많으므로 최빈값을 직접 구하는 것이 어렵습니다.
  - 간격이 일정한 계급을 설정하고, 해당 계급에 속한 도수와 상대도수를 표로 정리할 수 있습니다. [도수분포표]
- 매출의 최빈 구간은 평균보다 낮은 쪽에 있습니다.
  - 오른쪽으로 치우친 분포에서는 보통 **최빈값 < 중위수 < 평균** 순서가 됩니다.
- 연속형 변수의 최빈 구간은 구간 폭을 어떻게 정하느냐에 따라 달라지므로 참고용으로만 씁니다.

## 5. 분포 모양 확인: 오른쪽 치우침

**질문**: 점포 월평균 매출의 분포는 좌우 대칭인가?

In [ ]:
# 1행 2열의 그래프 영역을 생성합니다.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), dpi=100)

# 히스토그램을 그리고 평균과 중위수 위치에 세로선을 추가합니다.
sns.histplot(data=store_2025, x='sales', binrange=(0, 10000), bins=20, facecolor='silver', ax=axes[0])
axes[0].axvline(x=store_2025['sales'].mean(), c='red', label='평균')
axes[0].axvline(x=store_2025['sales'].median(), c='green', ls='--', label='중위수')
axes[0].set(title='점포 월평균 매출 히스토그램 (2025년)', xlabel='월평균 매출 (백만 원)', ylabel='점포 수')
axes[0].legend()

# 상자 그림을 그립니다.
sns.boxplot(data=store_2025, x='sales', boxprops={'facecolor': 'silver'}, medianprops={'color': 'red'}, ax=axes[1])
axes[1].set(title='점포 월평균 매출 상자 그림 (2025년)', xlabel='월평균 매출 (백만 원)', xlim=(-500, 10500))
plt.show()

**해석**
- 대부분의 점포가 왼쪽에 몰려 있고, 오른쪽으로 꼬리가 길게 늘어진 분포입니다.
  - 영어로 **right-skewed**이며, 오른쪽 치우친 분포라고 합니다.
- 상자 그림 오른쪽의 점들은 매출이 특히 큰 대형 점포입니다.
  - 수염은 경계 안쪽의 가장 작은 값과 가장 큰 값까지 그립니다.
  - 경계 하한보다 작거나, 경계 상한보다 큰 값을 동그란 점으로 표시합니다.
- 상자 그림에서 오른쪽 동그란 점에 해당하는 점포들이 평균을 중위수보다 오른쪽으로 끌고 갑니다.

**질문**: 치우친 정도를 숫자로 나타내면(왜도) 얼마인가?

In [ ]:
# 점포 월평균 매출의 왜도를 확인합니다.
print(f'왜도: {stats.skew(a=store_2025['sales']):.2f}')

In [ ]:
# 평균과 중위수를 비교합니다.
print(f'평균: {store_2025['sales'].mean():,.0f}')
print(f'중위수: {store_2025['sales'].median():,.0f}')

**해석**
- 왜도가 0이면 좌우 대칭, 양수면 오른쪽 꼬리가 긴 분포입니다.
  - 보통 왜도의 절댓값이 1을 넘으면 치우침이 큰 편으로 봅니다.
- 왜도가 양수이고 평균이 중위수보다 크다는 두 결과가 같은 이야기를 하고 있습니다.
  - 이런 분포에서 '보통 점포의 매출'을 말할 때는 평균보다 **중위수**가 더 적절합니다.

## 6. 대형 점포가 평균에 미치는 영향

**질문**: 매출 상위 3개 점포를 빼면 평균과 중위수는 각각 얼마나 바뀌는가?

In [ ]:
# 매출 상위 3개 점포의 ID를 top3_ids에 할당합니다.
top3_ids = store_2025.nlargest(n=3, columns='sales')['store_id']

# top3_ids를 출력합니다.
print(top3_ids)

In [ ]:
# 매출 상위 3개 점포를 제외한 데이터를 without_top3에 할당합니다.
without_top3 = store_2025.loc[~store_2025['store_id'].isin(values=top3_ids), :]

# 제외 전후 평균과 중위수를 비교하는 표를 생성하고 compare에 할당합니다.
compare = pd.DataFrame(data={'전체 100개': store_2025['sales'].agg(func=['mean', 'median']), 
                             '상위 3개 제외': without_top3['sales'].agg(func=['mean', 'median'])})

# compare의 행 이름을 평균, 중위수로 변경합니다.
compare.index = ['평균', '중위수']

# 매출 상위 3개 점포 제외 후의 변화율(%)을 추가합니다.
compare['변화율(%)'] = (compare['상위 3개 제외'] / compare['전체 100개'] - 1) * 100

# compare를 소수점 첫째 자리까지 반올림해서 확인합니다.
compare.round(decimals=1)

**해석**
- 매출 상위 3개 점포만 제외해도 평균이 중위수보다 더 많이 줄어듭니다.
  - 평균은 극단값 하나하나의 크기를 모두 반영합니다.
  - 중위수는 '가운데 순서'만 보기 때문에 극단값의 영향을 덜 받습니다. [**로버스트 통계량**]
- 보고서에 평균만 쓰면 소수 대형 점포 때문에 '일반 점포 수준'이 부풀려 보일 수 있습니다.

## 7. 로그 변환

**질문**: 매출에 로그를 취하면 분포 모양이 어떻게 바뀌는가?

In [ ]:
# 점포 월평균 매출에 로그를 취하고 log_sales에 할당합니다.
log_sales = np.log(store_2025['sales'])

# log_sales의 기술통계량을 확인합니다.
log_sales.describe().round(decimals=2)

In [ ]:
# 원 척도와 로그 척도의 왜도를 각각 계산합니다.
skew_sales = stats.skew(a=store_2025['sales'])
skew_log_sales = stats.skew(a=log_sales)

# 1행 2열의 그래프 영역을 생성합니다.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 4), dpi=100)

# 원 척도와 로그 척도의 커널 밀도 추정 곡선을 나란히 그리고 제목에 왜도를 표시합니다.
sns.kdeplot(data=store_2025, x='sales', color='silver', fill=True, ax=axes[0])
axes[0].set(title=f'원 척도 (왜도 {skew_sales:.2f})', xlabel='월평균 매출 (백만 원)', ylabel='밀도')

sns.kdeplot(x=log_sales, color='silver', fill=True, ax=axes[1])
axes[1].set(title=f'로그 척도 (왜도 {skew_log_sales:.2f})', xlabel='log(월평균 매출)', ylabel='밀도')
plt.show()

**해석**
- 매출·면적·금액처럼 '몇 배 차이'로 움직이는 비율형 변수는 로그 척도로 다루는 것이 좋습니다.
  - 0을 초과하는 비율형 변수에 로그를 취하면 큰 값은 많이 줄고 작은 값은 적게 줄어듭니다.
  - 그 결과 오른쪽 꼬리가 짧아지고 분포는 대칭에 가까워집니다. [로그 정규분포]
- 선형 회귀 실습에서 이 성질을 다시 사용합니다.

## 8. 데이터 표준화

**질문**: 단위가 다른 매출(백만 원)과 객단가(원) 중, 첫 번째 점포는 어느 쪽이 전체 점포 사이에서 더 튀는가?

In [ ]:
# 매출을 전체 점포의 평균과 표준편차로 표준화합니다.
store_2025['sales_z_all'] = (store_2025['sales'] - store_2025['sales'].mean()) / store_2025['sales'].std()

# 객단가를 전체 점포의 평균과 표준편차로 표준화합니다.
store_2025['ticket_z_all'] = (store_2025['avg_ticket'] - store_2025['avg_ticket'].mean()) / store_2025['avg_ticket'].std()

# 첫 번째 점포의 원래 값과 표준화 점수를 확인합니다.
store_2025.loc[[0], ['store_id', 'store_type', 'sales', 'sales_z_all', 'avg_ticket', 'ticket_z_all']] \
    .round(decimals=2)

In [ ]:
# 표준화 점수의 평균과 표준편차를 확인합니다.
store_2025.loc[:, ['sales_z_all', 'ticket_z_all']].agg(func=['mean', 'std']).round(decimals=2)

**해석**
- 매출과 객단가처럼 단위가 서로 다른 값의 크기를 직접 비교할 수 없습니다.
  - 단위가 서로 다른 값의 척도가 같아지도록 데이터를 정규화해야 합니다.
  - 데이터의 정규화 방식으로는 데이터 표준화, 최소-최대 정규화, 로버스트 정규화 등이 있습니다.
- 데이터 표준화는 원 데이터를 평균이 0, 표준편차가 1인 표준화 점수로 바꿉니다.
  - 표준화 점수는 **평균에서 표준편차 몇 개만큼 떨어져 있는가**를 나타냅니다.
  - 수능 표준점수가 과목별 난이도 차이를 없애기 위해 원점수 대신 표준점수를 비교하는 것과 같은 원리입니다.
  - 정규분포를 따르는 변수를 표준화하면 표준정규분포를 따릅니다.
- 표준화 점수가 0보다 크면 평균보다 위, 작으면 평균보다 아래입니다.
- 첫 번째 점포(S001)는 매출의 표준화 점수가 −0.62, 객단가의 표준화 점수가 −1.24입니다.
  - 두 값 모두 평균보다 아래이고, 객단가가 평균에서 더 멀리 떨어져 있으므로 객단가 쪽이 더 튑니다.

**질문**: 매출 3,000(백만 원)인 소형점과 4,500인 대형점 중, **자기 유형 안에서** 더 잘하고 있는 점포는 어디인가?

In [ ]:
# 점포 유형별 매출의 평균과 표준편차를 계산하고 type_stats에 할당합니다.
type_stats = store_2025.groupby(by='store_type')['sales'].agg(func=['mean', 'std'])

# type_stats를 소수점 첫째 자리까지 반올림해서 확인합니다.
type_stats.round(decimals=1)

In [ ]:
# 소형점 매출 3,000의 유형 안 표준화 점수를 계산하고 z_small에 할당합니다.
z_small = (3000 - type_stats.loc['소형', 'mean']) / type_stats.loc['소형', 'std']

# 소형점 매출 3,000의 표준화 점수를 확인합니다.
print('소형점 3,000:', round(number=z_small, ndigits=2))

In [ ]:
# 대형점 매출 4,500의 유형 안 표준화 점수를 계산하고 z_large에 할당합니다.
z_large = (4500 - type_stats.loc['대형', 'mean']) / type_stats.loc['대형', 'std']

# 대형점 매출 4,500의 표준화 점수를 확인합니다.
print('대형점 4,500:', round(number=z_large, ndigits=2))

**해석**
- 매출은 점포 규모에 비례하므로 대형점의 매출이 더 큽니다.
  - 점포 유형 안 표준화 점수로 비교하면 소형점에서 3,000(백만 원)은 평균보다 위이고, 대형점에서 4,500(백만 원)은 평균보다 아래입니다.
- 대형점은 원래 매출이 크고 퍼짐도 크기 때문에, 같은 1,000(백만 원) 차이라도 소형점에서 훨씬 큰 의미를 가집니다.

**질문**: 모든 점포의 매출을 유형 안에서 표준화하면, 유형 안에서 가장 잘하는 점포는 어디인가?

In [ ]:
# 점포 유형별로 묶은 매출을 grouped에 할당합니다.
grouped = store_2025.groupby(by='store_type')['sales']

# 점포 유형별 평균과 표준편차로 매출 표준화 점수를 계산합니다.
# [참고] transform()은 집단별 계산 결과를 원래 행 개수에 맞춰 돌려줍니다.
store_2025['sales_z_type'] = (store_2025['sales'] - grouped.transform(func='mean')) / grouped.transform(func='std')

# 점포 유형별 상위 점포 확인 관련 열 이름을 리스트로 생성합니다.
check_cols = ['store_id', 'store_type', 'sales', 'sales_z_type']

# 점포 유형별 표준화 점수 상위 점포를 확인합니다.
store_2025.loc[:, check_cols].sort_values(by='sales_z_type', ascending=False) \
    .groupby(by='store_type').first().round(decimals=2)

In [ ]:
# 점포 유형 안 표준화 점수 상위 5개 점포를 확인합니다.
store_2025.nlargest(n=5, columns='sales_z_type')[check_cols].round(decimals=2)

In [ ]:
# 원래 매출 상위 5개 점포를 확인합니다.
store_2025.nlargest(n=5, columns='sales')[check_cols].round(decimals=2)

**해석**
- 대형점이 매출 기준으로 높은 순위를 차지하지만, 점포 유형 안 표준화 점수 기준으로는 다른 유형의 점포도 상위에 올라옵니다.
- 점포 평가나 인센티브처럼 **규모가 다른 점포를 공정하게 비교**해야 할 때 점포 유형 안 표준화를 사용합니다.

## 9. 평균 매출의 95% 신뢰구간

**[생각해 보기] 점포 100개를 전부 가지고 있는데, 왜 '추정'을 하는가?**

- 2025년 실적을 **그대로 보고**하는 것이 목적이라면 현재 데이터는 모집단입니다.
  - 모집단의 평균은 추정값이 아니라 확정된 실적이고, 신뢰구간이 필요 없습니다.
- 실무에서 알고 싶은 것은 대개 '작년 우리 점포 100개의 실적'이 아니라, **이런 조건의 대형마트 점포라면 매출이 보통 어느 수준인가**입니다.
  - 신규 출점 계획을 세우거나 다른 해·다른 회사와 비교할 때가 그렇습니다.
  - 이 관점에서 100개 점포는 '대형마트 점포 일반'에서 나온 하나의 **표본**이고, 평균은 **점추정값**, 신뢰구간은 **구간추정값**입니다.
- 다음에 계산하는 신뢰구간이 추정하는 대상은 **우리 점포 100개의 2025년 실적 자체가 아닙니다.**
  - **이런 구조를 가진 대형마트 점포의 일반적인 평균 월매출**입니다.
- 정리하면, 숫자를 설명할 때는 모집단으로 보고, 그 숫자로 판단하거나 예측할 때는 **같은 생성 과정에서 나온 표본이라고 가정**하고 추정합니다.

**질문**: 표본평균, 표준오차, t 임계값, 오차한계를 차례로 계산해 95% 신뢰구간을 직접 구합니다.

In [ ]:
# 2025년 점포별 월평균 매출을 sales에 할당합니다.
sales = store_2025['sales']

# 표본 크기, 평균, 표본표준편차(n - 1로 나눔)를 계산합니다.
# [참고] n - 1로 나눈 표본분산은 모분산의 불편추정량입니다.
n, mean, sd = sales.shape[0], sales.mean(), sales.std()

# 표준오차를 계산합니다.
# [참고] 표준오차는 표본평균이 표본마다 흔들리는 정도입니다.
se = sd / np.sqrt(n)

# 양쪽 2.5%씩 자르는 t 임계값을 계산합니다.
t_crit = stats.t.ppf(q=0.975, df=n - 1)

# t 임계값에 표준오차를 곱해 오차한계를 계산합니다.
margin = t_crit * se

# 계산 결과와 95% 신뢰구간을 출력합니다.
print(f'n = {n}, 평균 = {mean:,.1f}, 표준편차 = {sd:,.1f}')
print(f'표준오차 = {se:,.1f}, t 임계값 = {t_crit:.3f}, 오차한계 = {margin:,.1f}')
print(f'95% 신뢰구간: {mean - margin:,.1f} ~ {mean + margin:,.1f}')

**해석**
- 세 값은 이름이 비슷하지만 뜻이 다릅니다.

  | 구분 | 뜻 | 관계 |
  |---|---|---|
  | **표준편차** | 개별 점포가 평균에서 얼마나 떨어져 있는지 | |
  | **표준오차** | 표본을 다시 뽑을 때마다 '표본평균'이 얼마나 흔들리는지 | 표준편차 ÷ $\sqrt{n}$이라 훨씬 작음 |
  | **오차한계** | 신뢰구간의 절반 폭 | t 임계값 × 표준오차 |

  [참고] 신뢰구간은 평균 ± 오차한계입니다.<br>
  [참고] 자유도가 작으면 t 임계값이 정규분포의 1.96보다 커서 구간이 조금 넓어집니다.

**질문**: `scipy.stats` 모듈로 계산한 결과도 직접 계산한 값과 같은가?

In [ ]:
# scipy.stats 모듈로 95% 신뢰구간을 계산합니다.
# [참고] stats.sem()은 표본평균의 표준오차를 반환합니다.
ci_lower, ci_upper = stats.t.interval(confidence=0.95, df=n - 1, loc=mean, scale=stats.sem(a=sales))

# scipy.stats 모듈로 계산한 95% 신뢰구간을 확인합니다.
print(f'scipy 95% 신뢰구간: {ci_lower:,.1f} ~ {ci_upper:,.1f}')

**해석**
- 두 결과가 같습니다.
- `stats.sem()`은 표준오차(standard error of the mean)를 계산합니다.
- 실무에서는 함수 한 줄로 계산하지만, 그 안에서 무슨 일이 일어나는지 알아야 결과를 설명할 수 있습니다.

**질문**: 점포 유형(대형·중형·소형)별로 나누어 평균 매출의 95% 신뢰구간을 계산하고 표로 정리하면 어떻게 되는가?

In [ ]:
# 점포 유형별 표본 크기, 평균, 표준편차, 표준오차를 계산하고 type_ci에 할당합니다.
# [참고] 'sem'은 stats.sem()과 같이 표본표준편차를 √n으로 나눈 표준오차를 계산합니다.
type_ci = store_2025.groupby(by='store_type')['sales'].agg(n='count', mean='mean', sd='std', se='sem')

# 점포 유형별 자유도에 맞는 t 임계값을 계산합니다.
type_ci['t_crit'] = stats.t.ppf(q=0.975, df=type_ci['n'] - 1)

# 점포 유형별 오차한계를 계산합니다.
type_ci['margin'] = type_ci['t_crit'] * type_ci['se']

# 95% 신뢰구간의 하한을 계산합니다.
type_ci['lower'] = type_ci['mean'] - type_ci['margin']

# 95% 신뢰구간의 상한을 계산합니다.
type_ci['upper'] = type_ci['mean'] + type_ci['margin']

# 소수점 첫째 자리까지 확인하되, t 임계값만 소수점 셋째 자리까지 확인합니다.
type_ci.sort_index().round(decimals=1).assign(t_crit=type_ci['t_crit'].round(decimals=3))

In [ ]:
# 1행 1열의 그래프 영역을 생성합니다.
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 4), dpi=100)

# 점포 유형별 평균과 95% 신뢰구간을 오차막대로 그립니다.
ax.errorbar(x=type_ci.index, y=type_ci['mean'], yerr=type_ci['margin'], fmt='o', color='black', ecolor='gray', capsize=8)

# 전체 점포 평균을 가로선으로 표시합니다.
ax.axhline(y=mean, color='red', linestyle='--', label='전체 점포 평균')

# 전체 점포 평균의 95% 신뢰구간을 분홍색 띠로 표시합니다.
ax.axhspan(ymin=mean - margin, ymax=mean + margin, color='pink', alpha=0.3, label='전체 점포 평균의 95% 신뢰구간')
ax.set(title='점포 유형별 월평균 매출의 95% 신뢰구간', xlabel='점포 유형', ylabel='월평균 매출 (백만 원)', xlim=(-0.5, 2.5))
ax.legend()
plt.show()

**해석**
- 세 유형의 신뢰구간이 서로 겹치지 않고 멀리 떨어져 있습니다.
  - 전체 평균은 어느 유형의 평균과도 거리가 있으므로 '보통 점포'를 대표한다고 보기 어렵습니다.
- 대형점은 소형점과 점포 수(n)가 같아도 표준편차가 커서 구간이 더 넓습니다.
  - 신뢰구간의 폭은 표본 크기와 표준편차가 함께 결정합니다.
- 현재 데이터를 유형별로 나누면 점포 수는 작아지지만 유형 안의 표준편차도 함께 작아집니다.
  - 중형·소형처럼 오차한계가 전체보다 오히려 좁아질 수 있습니다.
- 세 점포 유형의 신뢰구간이 겹치지 않는 것은 차이가 크다는 신호입니다.
  - 점포 유형 사이의 평균 차이를 검정하는 방법(ANOVA)은 가설검정 실습에서 다룹니다.

## 10. 신뢰구간을 업무 언어로 쓰기

**질문**: 신뢰구간을 보고서 문장으로 바꾸면 어떻게 쓰는가?

In [ ]:
# 신뢰구간을 보고서 문장으로 출력합니다.
print(f'점포 월평균 매출은 약 {ci_lower:,.0f}~{ci_upper:,.0f}백만 원 범위로 추정됩니다. (95% 신뢰수준)')

**해석**
- 위 문장처럼 '점포 월평균 매출은 약 ○○ ~ ○○백만 원 범위로 추정됩니다'라고 씁니다.
  - 신뢰구간의 정확한 의미는 '같은 방법으로 표본을 뽑아 구간을 만드는 일을 100번 반복하면, 그중 약 95번은 실제 평균을 포함한다'입니다.
- 이 구간은 점포 월평균 매출의 **모평균**에 대한 95% 신뢰구간이며, '점포 95%의 매출이 이 범위에 있다'는 뜻이 아닙니다.
  - 개별 점포의 매출은 위 히스토그램처럼 훨씬 넓게 퍼져 있습니다.
- 매출의 분포가 오른쪽으로 치우쳐 있으므로 보고서에는 중위수와 사분(위)범위를 함께 쓰는 것이 좋습니다.

**질문**: 점포 유형별 신뢰구간도 보고서 문장으로 바꾸면 어떻게 쓰는가?

In [ ]:
# 점포 유형별 신뢰구간을 보고서 문장으로 출력합니다.
for store_type, row in type_ci.iterrows():
    print(f'{store_type}점 월평균 매출은 약 {row['lower']:,.0f}~{row['upper']:,.0f}백만 원 범위로 추정됩니다. (95% 신뢰수준)')

**해석**
- 전체 평균 하나만 보고하면 유형 구성이 가려집니다.
- 보고서에는 전체 구간과 유형별 구간을 함께 표로 제시하는 것이 좋습니다.

## 정리

- `describe()`로 객단가의 범위를 확인해 매출 단위 입력 오류 행을 찾고, 원천 데이터 확인을 가정해 매출을 보정하고 객단가를 다시 계산했습니다.
  - 결합·보정한 데이터는 pickle 파일로 저장해 다른 실습에서 다시 사용합니다.
- 점포 × 월 행은 같은 점포가 반복되어 독립이 아닙니다.
  - 점포별 2025년 월평균 매출(n = 100)을 분석 단위로 삼았습니다.
- 범위는 극단값 두 개로 정해져 흔들리기 쉽고, 표준편차는 모든 값을 반영합니다.
  - 10분위수로 상위 10% 기준선과 상하위 격차(P90 / P10)를 확인했습니다.
- 범주형 변수는 `value_counts()`로 도수·상대도수를, `pd.crosstab()`으로 두 변수의 교차표를 만듭니다.
- 점포 매출은 오른쪽으로 치우쳐 평균이 중위수보다 크고, 소수 대형 점포가 평균을 크게 움직입니다.
  - 매출과 같은 비율형 변수를 로그 변환하면 분포가 대칭에 가까워집니다.
- 표준화 점수는 평균에서 표준편차 몇 개만큼 떨어져 있는지를 나타냅니다.
  - 점포 유형 안에서 표준화하면 규모가 다른 점포를 공정하게 비교할 수 있습니다.
- 평균 매출의 95% 신뢰구간을 t분포로 직접 계산했고, `scipy` 결과와 같음을 확인했습니다.
  - 점포 유형별 신뢰구간은 서로 겹치지 않을 만큼 떨어져 있어, 전체 평균 하나로는 점포 매출 수준을 대표하기 어렵습니다.
- 전체 데이터라도 실적을 기술할 때는 모집단, 같은 조건의 대형마트 점포 일반을 판단하거나 예측할 때는 같은 생성 과정에서 나온 표본이라고 가정합니다.
  - 표본으로 보면 평균은 점추정값, 신뢰구간은 구간추정값입니다.

## 현업 전달 포인트

통계 용어 없이 설명하면 다음과 같습니다.

- 점포 매출은 소수의 대형 점포가 크게 끌어올리는 구조라서, '보통 점포'의 매출은 평균보다 중위수로 보는 것이 정확합니다.
- 소형점 매출을 대형점과 그대로 비교하면 불리합니다.
  - 같은 유형 점포들의 평균과 퍼짐을 기준으로 환산한 점수로 비교하면, 규모와 상관없이 자기 체급에서 얼마나 잘하는지 볼 수 있습니다.
- 전체 점포의 평균 월매출 수준은 약 3,317 ~ 3,884백만 원으로 봅니다.
  - 점포 100개 자료로 계산한 범위라 어느 정도 오차를 감안한 값이며, 개별 점포가 모두 이 범위에 있다는 뜻은 아닙니다.
  - 다만 대형·중형·소형은 매출 수준이 확연히 다르므로 유형별 범위를 함께 보셔야 합니다.
- 원천 데이터에 매출 단위가 잘못 들어간 달이 있어, 원천 자료와 대조해 바로잡은 뒤 분석했습니다.
  - 시스템 입력 단계의 점검이 필요합니다.

## End of Document